# 14. Comparacion temporal de determinantes del ingreso laboral individual

**Pregunta.** ¿Como cambian entre 2018, 2020, 2022 y 2024 las asociaciones entre caracteristicas personales, laborales y territoriales y el ingreso laboral o de negocio trimestral de una persona adulta con ingreso positivo?

Cada fila es una persona; los cuatro levantamientos ENIGH son **cortes transversales repetidos**, no un panel. No se sigue a las mismas personas ni hogares. Los modelos separados describen cada corte; el modelo pooled compara promedios condicionados y las interacciones con año permiten probar diferencias de pendientes. Una variable significativa en un año y no en otro no prueba por si sola un cambio. Las asociaciones, incluidas las primas educativas observadas, no son efectos causales.

El target principal es **log1p del ingreso nominal individual trimestral**. Escolaridad se incorpora de forma trazable como predictor principal si pasa las validaciones; el nucleo sin escolaridad queda para comparar sensibilidad. La geografia principal son las regiones Banxico; las entidades federativas son una sensibilidad separada. La inferencia usa toda la muestra valida por año, sin seleccion por significancia ni holdout. El notebook 13 sigue siendo el diagnostico detallado local de 2024. Este notebook solo lee cuatro bases individuales ya exportadas por el 12 y no usa bases de hogares.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from scipy.stats import norm

ANIOS = [2018, 2020, 2022, 2024]
ANIO_REFERENCIA = 2024
ESPECIFICACION_GEOGRAFICA_PRINCIPAL = "region_banxico"
EJECUTAR_ESPECIFICACION_ENTIDAD = True
MOSTRAR_SUMMARIES_COMPLETOS = True
EXPORTAR_RESULTADOS = True
EJECUTAR_MODELOS_PONDERADOS = True
EJECUTAR_INTERACCIONES_COMPLETAS = True
TARGET_PRINCIPAL = "log_ingreso_nominal"
EJECUTAR_TARGET_REAL_APROX = False
TARGET_REAL_APROX_ES_SENSIBILIDAD = True
COVARIANZA_INFERENCIA_PRINCIPAL = "cluster_hogar"
COVARIANZA_SENSIBILIDAD = "HC3"

# Solo se usan si EJECUTAR_INTERACCIONES_COMPLETAS es False.
BLOQUES_INTERACCION = ["individuales", "laborales", "localidad", "region"]
VARIABLES_INTERACCION_MANUAL = []
EJECUTAR_ESPECIFICACION_AMPLIADA = True
ALFA = 0.05
EDAD_MINIMA = 18

if ANIOS != [2018, 2020, 2022, 2024] or ANIO_REFERENCIA not in ANIOS:
    raise ValueError("Revisar años y referencia temporal.")
if ESPECIFICACION_GEOGRAFICA_PRINCIPAL != "region_banxico":
    raise ValueError("El contrato fija region_banxico como especificacion principal.")
if not 0 < ALFA < 1:
    raise ValueError("ALFA debe estar entre cero y uno.")
if TARGET_PRINCIPAL != "log_ingreso_nominal" or not TARGET_REAL_APROX_ES_SENSIBILIDAD:
    raise ValueError("El log nominal es principal; el real aproximado solo es sensibilidad.")
if COVARIANZA_INFERENCIA_PRINCIPAL != "cluster_hogar" or COVARIANZA_SENSIBILIDAD != "HC3":
    raise ValueError("Revisar la jerarquia de covarianzas declarada.")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").exists():
    raise FileNotFoundError("Abrir Jupyter desde la raiz del proyecto o notebooks/.")

KEY_COLS = ["anio", "folioviv", "foliohog", "numren"]
HOGAR_COLS = KEY_COLS[:3]
TARGET_NOMINAL = "ingreso_persona_laboral_negocio_tri"
LOG_NOMINAL = "log1p_ingreso_persona_laboral_negocio_tri"
variables_comunes_numericas = ["edad", "n_trabajos", "horas_trabajos_total"]
variables_comunes_categoricas = [
    "sexo_desc", "hablaind_desc", "segsoc_desc", "subor_principal_desc",
    "contrato_principal_desc", "tam_loc_desc",
]
variable_geografica_regional = ["region_banxico"]
variable_geografica_estatal = ["entidad"]
variables_pendientes = [
    "parentesco_desc", "tam_emp_principal_desc", "est_socio_desc",
]
VARIABLE_EDUCACION = "nivel_educativo_homologado"
ESTRUCTURALES = {
    "subor_principal_desc": "No aplica: sin trabajo principal reportado",
    "contrato_principal_desc": "No aplica: sin contrato principal documentado",
}
REFERENCIAS = {
    "sexo_desc": "Hombre",
    "hablaind_desc": "No",
    "segsoc_desc": "No",
    "subor_principal_desc": "No",
    "contrato_principal_desc": "No",
    "tam_loc_desc": "Localidades con 100 000 y más habitantes",
    "region_banxico": "Centro",
    "entidad": "Ciudad de Mexico",
    VARIABLE_EDUCACION: "Secundaria",
}
BLOQUES = {
    "individuales": ["edad", "sexo_desc", "hablaind_desc"],
    "laborales": ["n_trabajos", "horas_trabajos_total", "segsoc_desc",
                  "subor_principal_desc", "contrato_principal_desc"],
    "localidad": ["tam_loc_desc"],
    "region": ["region_banxico"],
    "escolaridad": [VARIABLE_EDUCACION],
}
INPUTS = {
    anio: PROJECT_ROOT / "data" / "processed" / f"determinantes_{anio}"
          / f"base_interpretable_personas_{anio}.csv.gz"
    for anio in ANIOS
}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "comparacion_temporal_determinantes"
display(pd.DataFrame([
    {"parametro": k, "valor": str(v)} for k, v in {
        "ANIOS": ANIOS, "ANIO_REFERENCIA": ANIO_REFERENCIA,
        "ESPECIFICACION_GEOGRAFICA_PRINCIPAL": ESPECIFICACION_GEOGRAFICA_PRINCIPAL,
        "EJECUTAR_ESPECIFICACION_ENTIDAD": EJECUTAR_ESPECIFICACION_ENTIDAD,
        "MOSTRAR_SUMMARIES_COMPLETOS": MOSTRAR_SUMMARIES_COMPLETOS,
        "EXPORTAR_RESULTADOS": EXPORTAR_RESULTADOS,
        "EJECUTAR_MODELOS_PONDERADOS": EJECUTAR_MODELOS_PONDERADOS,
        "EJECUTAR_INTERACCIONES_COMPLETAS": EJECUTAR_INTERACCIONES_COMPLETAS,
        "TARGET_PRINCIPAL": TARGET_PRINCIPAL,
        "EJECUTAR_TARGET_REAL_APROX": EJECUTAR_TARGET_REAL_APROX,
        "TARGET_REAL_APROX_ES_SENSIBILIDAD": TARGET_REAL_APROX_ES_SENSIBILIDAD,
        "COVARIANZA_INFERENCIA_PRINCIPAL": COVARIANZA_INFERENCIA_PRINCIPAL,
        "COVARIANZA_SENSIBILIDAD": COVARIANZA_SENSIBILIDAD,
        "EJECUTAR_ESPECIFICACION_AMPLIADA": EJECUTAR_ESPECIFICACION_AMPLIADA,
        "ALFA": ALFA, "EDAD_MINIMA": EDAD_MINIMA,
    }.items()
]))

## 1. Lectura y auditoria de las cuatro bases

Se validan archivos, llaves, año, edad, ingreso positivo, transformaciones, geografia y variables de diseño. Se construyen identificadores explicitos de persona-año y hogar-año; los conteos mostrados son agregados y no se imprimen folios. Los targets reales tienen sufijo del **año de origen** en el archivo, aunque el factor los expresa aproximadamente en pesos de 2024. La consistencia aritmetica no valida por si misma la pertinencia economica del factor.

In [ ]:
def nombres_targets(anio):
    return {
        "nivel_nominal": TARGET_NOMINAL,
        "log_nominal": LOG_NOMINAL,
        "nivel_real_2024": f"ingreso_persona_laboral_negocio_tri_real_{anio}",
        "log_real_2024": f"log1p_ingreso_persona_laboral_negocio_tri_real_{anio}",
    }


def auditar_base(anio, ruta):
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    base = pd.read_csv(ruta, low_memory=False)
    targets = nombres_targets(anio)
    obligatorias = (
        KEY_COLS + list(targets.values()) + variables_comunes_numericas
        + variables_comunes_categoricas + variable_geografica_regional
        + variable_geografica_estatal + variables_pendientes + ["nivelaprob_desc"]
        + ["factor", "est_dis", "upm", "cve_ent", "deflactor_2024",
           "segsoc_desc_original"]
    )
    ausentes = sorted(set(obligatorias) - set(base.columns))
    if ausentes:
        raise KeyError(f"{anio}: columnas ausentes: {ausentes}")
    nominal = pd.to_numeric(base[TARGET_NOMINAL], errors="coerce")
    real = pd.to_numeric(base[targets["nivel_real_2024"]], errors="coerce")
    log_nominal = pd.to_numeric(base[LOG_NOMINAL], errors="coerce")
    log_real = pd.to_numeric(base[targets["log_real_2024"]], errors="coerce")
    deflactor = pd.to_numeric(base["deflactor_2024"], errors="coerce")
    edad = pd.to_numeric(base["edad"], errors="coerce")
    if base.empty or base[KEY_COLS].isna().any().any():
        raise ValueError(f"{anio}: base vacia o llaves incompletas.")
    if not base["anio"].eq(anio).all() or base.duplicated(KEY_COLS).any():
        raise ValueError(f"{anio}: año incorrecto o llave personal duplicada.")
    base["id_hogar_anio"] = pd.MultiIndex.from_frame(base[HOGAR_COLS]).to_flat_index()
    base["id_persona_anio"] = pd.MultiIndex.from_frame(base[KEY_COLS]).to_flat_index()
    if base.id_persona_anio.nunique() != len(base):
        raise ValueError(f"{anio}: id_persona_anio no es unico.")
    if base.id_hogar_anio.nunique() != base[HOGAR_COLS].drop_duplicates().shape[0]:
        raise ValueError(f"{anio}: colision de id_hogar_anio.")
    if not edad.ge(EDAD_MINIMA).all() or not (nominal.gt(0) & np.isfinite(nominal)).all():
        raise ValueError(f"{anio}: edad/ingreso fuera del universo.")
    if not (real.gt(0) & np.isfinite(real)).all() or not (
        deflactor.gt(0) & np.isfinite(deflactor)
    ).all() or deflactor.nunique() != 1:
        raise ValueError(f"{anio}: target real o factor invalido.")
    if anio == ANIO_REFERENCIA and not np.isclose(deflactor.iloc[0], 1):
        raise ValueError("El factor del año de referencia debe ser uno.")
    if not (
        np.allclose(log_nominal, np.log1p(nominal), rtol=1e-10, atol=1e-10)
        and np.allclose(real, nominal * deflactor, rtol=1e-10, atol=1e-10)
        and np.allclose(log_real, np.log1p(real), rtol=1e-10, atol=1e-10)
    ):
        raise ValueError(f"{anio}: targets o transformaciones inconsistentes.")
    if base[["region_banxico", "entidad", "cve_ent"]].isna().any().any():
        raise ValueError(f"{anio}: geografia incompleta.")
    codigos = base[["cve_ent", "entidad"]].drop_duplicates()
    if (
        codigos.cve_ent.nunique() != codigos.entidad.nunique()
        or codigos.duplicated("cve_ent").any()
        or codigos.duplicated("entidad").any()
    ):
        raise ValueError(f"{anio}: codigo y nombre estatal no son uno a uno.")
    peso = pd.to_numeric(base["factor"], errors="coerce")
    diseno_valido = bool(
        peso.gt(0).all() and np.isfinite(peso).all()
        and base[["est_dis", "upm"]].notna().all().all()
        and base["est_dis"].nunique() > 1 and base["upm"].nunique() > 1
    )
    fila = {
        "anio": anio, "archivo": str(ruta.relative_to(PROJECT_ROOT)),
        "n_personas": len(base),
        "n_hogares": base.id_hogar_anio.nunique(),
        "duplicados_persona": int(base.duplicated(KEY_COLS).sum()),
        "edad_minima_observada": float(edad.min()),
        "ingreso_nominal_minimo": float(nominal.min()),
        "deflactor_2024": float(deflactor.iloc[0]),
        "regiones": base.region_banxico.nunique(),
        "entidades": base.entidad.nunique(),
        "factor_valido_y_diseno_disponible": diseno_valido,
        "faltantes_factor": int(peso.isna().sum()),
        "faltantes_est_dis": int(base.est_dis.isna().sum()),
        "faltantes_upm": int(base.upm.isna().sum()),
    }
    return base, fila


bases = {}
filas_auditoria = []
for anio in ANIOS:
    bases[anio], fila = auditar_base(anio, INPUTS[anio])
    filas_auditoria.append(fila)
ids_temporales = pd.concat([base[["anio", "id_hogar_anio", "id_persona_anio"]] for base in bases.values()], ignore_index=True)
if ids_temporales.id_persona_anio.duplicated().any():
    raise ValueError("Colision de persona-año entre cortes.")
if ids_temporales.groupby("id_hogar_anio").anio.nunique().gt(1).any():
    raise ValueError("Un hogar-año se comparte entre años distintos.")
if ids_temporales.groupby("anio").id_hogar_anio.nunique().to_dict() != dict(zip(ANIOS, [fila["n_hogares"] for fila in filas_auditoria])):
    raise ValueError("Conteos de hogares-año inconsistentes.")
codigos_estatales = pd.concat([
    base[["cve_ent", "entidad"]].drop_duplicates() for base in bases.values()
], ignore_index=True).drop_duplicates()
if (codigos_estatales.groupby("cve_ent").entidad.nunique().gt(1).any()
    or codigos_estatales.groupby("entidad").cve_ent.nunique().gt(1).any()):
    raise ValueError("Codigo/nombre estatal cambia entre años.")
auditoria_bases_temporales = pd.DataFrame(filas_auditoria)
display(auditoria_bases_temporales)

## 2. Contrato comun, faltantes y referencias

Nucleo: tres predictores numericos y seis categoricos; la region se añade por separado. Los faltantes de subordinacion y contrato corresponden a rutas estructurales del cuestionario y reciben exactamente las etiquetas usadas en el notebook 13. Los faltantes originales permanecen en la auditoria. No se reagrupan categorias raras. Las referencias se fijan una vez; si falta una, se muestra la interseccion candidata y se detiene el ajuste. La codificacion de seguridad social se verifica contra el original conservado por el 12.

Escolaridad no entraba en el nucleo por una limitacion tecnica provisional de comparabilidad, no por irrelevancia sustantiva. Su omision puede alterar las asociaciones estimadas de edad, sexo, contrato, subordinacion, seguridad social y geografia. La seccion siguiente audita codigos y etiquetas antes de preparar el ampliado. Parentesco queda como sensibilidad futura. Tamaño de empresa y estrato socioeconomico requieren revisar dependencia laboral y nivel conceptual, respectivamente.

In [ ]:
SEGSOC_ORIGEN_ESPERADO = {
    2018: {"No": "No", "Sí": "Sí"},
    2020: {"No nóicpircseD .0202 y": "No", "Sí": "Sí"},
    2022: {"No nóicpircseD .2202 y": "No", "Sí": "Sí"},
    2024: {"No": "No", "Sí": "Sí"},
}
bases_modelo = {}
for anio, base in bases.items():
    pares = base[["segsoc_desc_original", "segsoc_desc"]].drop_duplicates()
    observados = dict(zip(pares.segsoc_desc_original, pares.segsoc_desc))
    if observados != SEGSOC_ORIGEN_ESPERADO[anio]:
        raise ValueError(f"{anio}: etiquetas de segsoc_desc inesperadas: {observados}")
    preparada = base.copy(deep=False)
    for variable, etiqueta in ESTRUCTURALES.items():
        preparada[variable] = preparada[variable].fillna(etiqueta)
    bases_modelo[anio] = preparada

variables_comparar = (
    variables_comunes_numericas + variables_comunes_categoricas
    + variable_geografica_regional + variable_geografica_estatal
    + ["nivelaprob_desc"] + variables_pendientes
)
filas_comparabilidad = []
for variable in variables_comparar:
    numerica = variable in variables_comunes_numericas
    geografica = variable in variable_geografica_regional + variable_geografica_estatal
    pendiente = variable in variables_pendientes
    fila = {
        "variable": variable,
        "tipo": "numerica" if numerica else "categorica",
        "nivel_conceptual": (
            "individual" if variable in BLOQUES["individuales"]
            or variable == "nivelaprob_desc" else
            "laboral" if variable in BLOQUES["laborales"]
            or variable == "tam_emp_principal_desc" else
            "territorial" if geografica or variable == "tam_loc_desc" else
            "contexto_hogar" if variable == "est_socio_desc" else "parentesco"
        ),
        "referencia_comun": REFERENCIAS.get(variable),
        "incluida_nucleo": not pendiente and variable != "nivelaprob_desc",
        "incluida_ampliada": False,
        "pendiente": pendiente or variable == "nivelaprob_desc",
        "regla_armonizacion": (
            "2020/2022: correccion exacta de segsoc_desc validada con original"
            if variable == "segsoc_desc" else
            "faltante estructural con etiqueta explicita"
            if variable in ESTRUCTURALES else
            "homologacion trazable en nivel_educativo_homologado"
            if variable == "nivelaprob_desc" else
            "sin recodificacion; exige homologacion semantica"
            if pendiente else "misma definicion provisional"
        ),
        "motivo": (
            "codigo no exportado; mapeo por etiqueta exacta y catalogo oficial"
            if variable == "nivelaprob_desc" else
            "etiquetas y categorias difieren; sensibilidad futura"
            if variable == "parentesco_desc" else
            "dependencia con subordinacion y faltante estructural"
            if variable == "tam_emp_principal_desc" else
            "estrato del hogar; revisar nivel conceptual"
            if variable == "est_socio_desc" else ""
        ),
    }
    for anio in ANIOS:
        base = bases[anio]
        preparada = bases_modelo[anio]
        disponible = variable in base
        fila[f"disponible_{anio}"] = disponible
        fila[f"faltantes_{anio}"] = int(base[variable].isna().sum()) if disponible else None
        fila[f"faltantes_pct_{anio}"] = float(base[variable].isna().mean() * 100) if disponible else None
        if disponible and not numerica:
            conteos = preparada[variable].value_counts(dropna=False)
            fila[f"categorias_{anio}"] = " | ".join(sorted(map(str, conteos.index)))
            fila[f"frecuencia_minima_{anio}"] = int(conteos.min())
        else:
            fila[f"categorias_{anio}"] = None
            fila[f"frecuencia_minima_{anio}"] = None
    filas_comparabilidad.append(fila)

comparabilidad_variables = pd.DataFrame(filas_comparabilidad)
display(comparabilidad_variables)
display(comparabilidad_variables.loc[
    comparabilidad_variables.variable.isin(variables_pendientes),
    ["variable", "referencia_comun", "motivo"]
    + [f"categorias_{anio}" for anio in ANIOS]
])

In [ ]:
CATEGORIAS_COMUNES = {}
filas_referencias = []
for variable in variables_comunes_categoricas + ["region_banxico", "entidad"]:
    conjuntos = {
        anio: set(bases_modelo[anio][variable].dropna().astype(str))
        for anio in ANIOS
    }
    referencia = REFERENCIAS[variable]
    interseccion = set.intersection(*conjuntos.values())
    iguales = all(c == conjuntos[ANIO_REFERENCIA] for c in conjuntos.values())
    for anio in ANIOS:
        filas_referencias.append({
            "anio": anio, "variable": variable, "referencia": referencia,
            "referencia_presente": referencia in conjuntos[anio],
            "categorias": " | ".join(sorted(conjuntos[anio])),
            "ausentes_vs_2024": " | ".join(sorted(conjuntos[ANIO_REFERENCIA] - conjuntos[anio])),
            "nuevas_vs_2024": " | ".join(sorted(conjuntos[anio] - conjuntos[ANIO_REFERENCIA])),
            "categorias_iguales_cuatro_anios": iguales,
        })
    if not iguales or referencia not in interseccion:
        print(pd.DataFrame(filas_referencias).loc[lambda d: d.variable.eq(variable)])
        raise ValueError(
            f"{variable}: categorias/referencia incompatibles. "
            f"Referencias comunes candidatas (no seleccionadas): {sorted(interseccion)}"
        )
    CATEGORIAS_COMUNES[variable] = [referencia] + sorted(conjuntos[ANIO_REFERENCIA] - {referencia})
referencias_categoricas = pd.DataFrame(filas_referencias)
display(referencias_categoricas[["anio", "variable", "referencia",
                                "referencia_presente", "categorias_iguales_cuatro_anios"]])

### Escolaridad: auditoria y homologacion trazable

Escolaridad es central para la pregunta sustantiva. El nucleo sin ella se conserva para medir sensibilidad de las otras betas, no como conclusion definitiva. Las bases exportadas incluyen **nivelaprob_desc**, pero no el codigo **nivelaprob**. En una auditoria de lectura previa, las etiquetas se cotejaron por llave de persona con poblacion.csv y cada etiqueta correspondio a un solo codigo en su año; el notebook **reconstruye el codigo desde la etiqueta exacta y el catalogo oficial**, sin afirmar que la base lo contiene ni leer microdatos adicionales.

Los [descriptores oficiales INEGI 2018](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2018/microdatos/702825188061.pdf), [2020](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2020/microdatos/889463901242.pdf), [2022](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/microdatos/889463910626.pdf) y [2024](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/889463924494.pdf) documentan nivelaprob. El codigo 08 en 2024 es Especialidad; Maestria y Doctorado pasan a 09 y 10. La [SEP clasifica Especialidad, Maestria y Doctorado como posgrado](https://educacionsuperior.sep.gob.mx/Instituciones-SES/Instituciones-UPs). Se agregan en un grupo amplio **Posgrado**: no se equiparan sus diplomas ni se afirma que la categoria Especialidad existia antes de 2024.

Normal y Tecnica/comercial permanecen grupos separados: su ubicacion educativa exacta no se infiere del nombre. Profesional y Licenciatura/Ingenieria se agrupan en una clase amplia con el cambio de texto documentado. Ninguno y Preescolar se agrupan como instruccion nula/inicial. La referencia comun propuesta es Secundaria por significado estable y frecuencia suficiente. Toda etiqueta inesperada detiene el notebook; no hay grupo "otros".

In [ ]:
FUENTES_EDUCACION = {
    2018: "INEGI doc_2018.pdf, p. 74, nivelaprob; https://www.inegi.org.mx/contenidos/programas/enigh/nc/2018/microdatos/702825188061.pdf",
    2020: "INEGI doc_2020.pdf, p. 79, nivelaprob; https://www.inegi.org.mx/contenidos/programas/enigh/nc/2020/microdatos/889463901242.pdf",
    2022: "INEGI doc_2022.pdf, p. 79, nivelaprob; https://www.inegi.org.mx/contenidos/programas/enigh/nc/2022/microdatos/889463910626.pdf",
    2024: "INEGI doc_2024.pdf, p. 89, nivelaprob; https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/889463924494.pdf",
}
FUENTE_POSGRADO = "SEP: https://educacionsuperior.sep.gob.mx/Instituciones-SES/Instituciones-UPs"
# Codigo, etiqueta exacta en la base, nivel comparable y regla sustantiva.
CATALOGO_2018_2022 = [
    ("0", "Ninguno", "Sin escolaridad/inicial", "agregacion con preescolar"),
    ("1", "Preescolar", "Sin escolaridad/inicial", "agregacion de instruccion inicial"),
    ("2", "Primaria", "Primaria", "misma categoria"),
    ("3", "Secundaria", "Secundaria", "misma categoria"),
    ("4", "Preparatoria o bachillerato", "Media superior", "grupo amplio de bachillerato"),
    ("5", "Normal", "Normal", "categoria propia; no presumir equivalencia con licenciatura"),
    ("6", "Carrera técnica o comercial", "Tecnica/comercial", "categoria propia; cambio de texto en 2024"),
    ("7", "Profesional", "Licenciatura/profesional", "grupo amplio; cambio de texto en 2024"),
    ("8", "Maestría", "Posgrado", "agregacion de posgrado"),
    ("9", "Doctorado", "Posgrado", "agregacion de posgrado"),
]
CATALOGO_2024 = [
    ("00", "Ninguno", "Sin escolaridad/inicial", "agregacion con preescolar"),
    ("01", "Preescolar o kínder", "Sin escolaridad/inicial", "cambio de texto; instruccion inicial"),
    ("02", "Primaria", "Primaria", "misma categoria"),
    ("03", "Secundaria", "Secundaria", "misma categoria"),
    ("04", "Preparatoria o bachillerato", "Media superior", "grupo amplio de bachillerato"),
    ("05", "Normal", "Normal", "categoria propia; no presumir equivalencia con licenciatura"),
    ("06", "Estudios técnicos o comerciales", "Tecnica/comercial", "categoria propia; cambio de texto"),
    ("07", "Licenciatura o Ingeniería (profesional)", "Licenciatura/profesional", "grupo amplio; cambio de texto"),
    ("08", "Especialidad", "Posgrado", "nueva desagregacion 2024; posgrado segun SEP"),
    ("09", "Maestría", "Posgrado", "codigo desplazado por Especialidad"),
    ("10", "Doctorado", "Posgrado", "codigo desplazado por Especialidad"),
]
filas_mapeo = []
for anio in ANIOS:
    catalogo = CATALOGO_2024 if anio == 2024 else CATALOGO_2018_2022
    for codigo, etiqueta, nivel, regla in catalogo:
        filas_mapeo.append({
            "anio": anio, "variable_original": "nivelaprob_desc",
            "codigo_original_catalogo": codigo, "etiqueta_original": etiqueta,
            "nivel_educativo_homologado": nivel, "regla_aplicada": regla,
            "fuente": FUENTES_EDUCACION[anio],
            "justificacion_adicional": FUENTE_POSGRADO if nivel == "Posgrado" else "",
            "codigo_en_base_exportada": False,
        })
mapeo_escolaridad_homologada = pd.DataFrame(filas_mapeo)
if (mapeo_escolaridad_homologada.duplicated(["anio", "etiqueta_original"]).any()
    or mapeo_escolaridad_homologada.duplicated(["anio", "codigo_original_catalogo"]).any()):
    raise ValueError("Catalogo educativo ambiguo: codigo o etiqueta duplicados.")
filas_auditoria_educacion = []
filas_frecuencias_educacion = []
filas_validacion_educacion = []
for anio in ANIOS:
    base = bases[anio]
    catalogo = mapeo_escolaridad_homologada.loc[
        mapeo_escolaridad_homologada.anio.eq(anio)
    ]
    etiquetas = set(base.nivelaprob_desc.dropna())
    desconocidas = etiquetas - set(catalogo.etiqueta_original)
    if desconocidas:
        raise ValueError(f"{anio}: etiquetas educativas no contempladas: {sorted(desconocidas)}")
    por_etiqueta = catalogo.set_index("etiqueta_original")
    pesos = pd.to_numeric(base.factor, errors="coerce")
    conteos = base.groupby("nivelaprob_desc", dropna=False).size()
    pesos_etiqueta = pesos.groupby(base.nivelaprob_desc, dropna=False).sum()
    for fila in catalogo.to_dict(orient="records"):
        etiqueta = fila["etiqueta_original"]
        filas_auditoria_educacion.append({
            **fila, "frecuencia": int(conteos.get(etiqueta, 0)),
            "frecuencia_ponderada": float(pesos_etiqueta.get(etiqueta, 0)),
            "faltantes_originales_anio": int(base.nivelaprob_desc.isna().sum()),
            "categoria_presente": etiqueta in etiquetas,
            "categoria_inesperada": False,
            "observacion": fila["regla_aplicada"],
        })
    bases_modelo[anio]["nivelaprob_codigo_catalogo"] = (
        base.nivelaprob_desc.map(por_etiqueta.codigo_original_catalogo)
    )
    bases_modelo[anio][VARIABLE_EDUCACION] = (
        base.nivelaprob_desc.map(por_etiqueta.nivel_educativo_homologado)
    )
    if bases_modelo[anio][VARIABLE_EDUCACION].isna().sum() != base.nivelaprob_desc.isna().sum():
        raise ValueError(f"{anio}: perdida de filas por recodificacion educativa.")
    if not bases_modelo[anio].nivelaprob_desc.equals(base.nivelaprob_desc):
        raise ValueError(f"{anio}: se altero la etiqueta educativa original.")
    for nivel, grupo in bases_modelo[anio].groupby(VARIABLE_EDUCACION, dropna=False):
        filas_frecuencias_educacion.append({
            "anio": anio, "nivel_educativo_homologado": nivel,
            "frecuencia": len(grupo),
            "frecuencia_ponderada": float(pd.to_numeric(grupo.factor).sum()),
        })
    filas_validacion_educacion.append({
        "anio": anio, "etiquetas_no_faltantes_cubiertas": len(desconocidas) == 0,
        "faltantes_originales": int(base.nivelaprob_desc.isna().sum()),
        "filas_conservadas": len(bases_modelo[anio]) == len(base),
        "codigos_reconstruidos": int(bases_modelo[anio].nivelaprob_codigo_catalogo.notna().sum()),
        "referencia_presente": bool(bases_modelo[anio][VARIABLE_EDUCACION].eq("Secundaria").any()),
        "frecuencia_referencia": int(bases_modelo[anio][VARIABLE_EDUCACION].eq("Secundaria").sum()),
        "frecuencia_minima_nivel": int(bases_modelo[anio][VARIABLE_EDUCACION].value_counts().min()),
        "codigo_etiqueta_univoco_catalogo": bool(
            not catalogo.duplicated("codigo_original_catalogo").any()
            and not catalogo.duplicated("etiqueta_original").any()
        ),
    })
auditoria_escolaridad_por_anio = pd.DataFrame(filas_auditoria_educacion)
frecuencias_escolaridad_homologada = pd.DataFrame(filas_frecuencias_educacion)
validaciones_escolaridad = pd.DataFrame(filas_validacion_educacion)
conjuntos_educativos = {
    anio: set(bases_modelo[anio][VARIABLE_EDUCACION].dropna()) for anio in ANIOS
}
etiquetas_por_anio = {
    anio: set(bases[anio].nivelaprob_desc.dropna()) for anio in ANIOS
}
for anio in ANIOS:
    fila_anio = validaciones_escolaridad.anio.eq(anio)
    validaciones_escolaridad.loc[fila_anio, "etiquetas_ausentes_vs_2024"] = (
        " | ".join(sorted(etiquetas_por_anio[ANIO_REFERENCIA] - etiquetas_por_anio[anio]))
    )
    validaciones_escolaridad.loc[fila_anio, "etiquetas_nuevas_vs_2024"] = (
        " | ".join(sorted(etiquetas_por_anio[anio] - etiquetas_por_anio[ANIO_REFERENCIA]))
    )
    validaciones_escolaridad.loc[fila_anio, "niveles_ausentes_vs_2024"] = (
        " | ".join(sorted(conjuntos_educativos[ANIO_REFERENCIA] - conjuntos_educativos[anio]))
    )
categorias_educativas_iguales = all(
    categorias == conjuntos_educativos[ANIO_REFERENCIA]
    for categorias in conjuntos_educativos.values()
)
HOMOLOGACION_APTA = bool(
    categorias_educativas_iguales
    and validaciones_escolaridad[
        ["etiquetas_no_faltantes_cubiertas", "filas_conservadas", "referencia_presente"]
    ].all().all()
    and validaciones_escolaridad.frecuencia_referencia.ge(1000).all()
    and validaciones_escolaridad.frecuencia_minima_nivel.ge(30).all()
    and validaciones_escolaridad.codigo_etiqueta_univoco_catalogo.all()
)
if HOMOLOGACION_APTA:
    CATEGORIAS_COMUNES[VARIABLE_EDUCACION] = [
        "Secundaria"
    ] + sorted(conjuntos_educativos[ANIO_REFERENCIA] - {"Secundaria"})
    referencias_categoricas = pd.concat([
        referencias_categoricas,
        pd.DataFrame([{
            "anio": anio, "variable": VARIABLE_EDUCACION,
            "referencia": "Secundaria", "referencia_presente": True,
            "categorias": " | ".join(CATEGORIAS_COMUNES[VARIABLE_EDUCACION]),
            "ausentes_vs_2024": "", "nuevas_vs_2024": "",
            "categorias_iguales_cuatro_anios": True,
        } for anio in ANIOS])
    ], ignore_index=True)
estado_escolaridad = (
    "homologacion_preparada_sujeta_a_validacion_manual"
    if HOMOLOGACION_APTA else "bloqueada; solo nucleo preliminar"
)
EJECUTAR_AMPLIADO = EJECUTAR_ESPECIFICACION_AMPLIADA and HOMOLOGACION_APTA
comparabilidad_variables.loc[
    comparabilidad_variables.variable.eq("nivelaprob_desc"), "pendiente"
] = not HOMOLOGACION_APTA
comparabilidad_variables = pd.concat([
    comparabilidad_variables,
    pd.DataFrame([{
        "variable": VARIABLE_EDUCACION, "tipo": "categorica",
        "nivel_conceptual": "individual", "referencia_comun": "Secundaria",
        "incluida_nucleo": False, "incluida_ampliada": EJECUTAR_AMPLIADO,
        "pendiente": not HOMOLOGACION_APTA,
        "regla_armonizacion": "catalogos oficiales por año y agrupacion educativa explicita",
        "motivo": "variable sustantiva central; codigos reconstruidos desde etiquetas exactas",
        **{
            clave: valor for anio in ANIOS for clave, valor in {
                f"disponible_{anio}": True,
                f"faltantes_{anio}": int(validaciones_escolaridad.loc[
                    validaciones_escolaridad.anio.eq(anio), "faltantes_originales"
                ].iloc[0]),
                f"faltantes_pct_{anio}": float(
                    bases[anio].nivelaprob_desc.isna().mean() * 100
                ),
                f"categorias_{anio}": " | ".join(sorted(conjuntos_educativos[anio])),
                f"frecuencia_minima_{anio}": int(
                    frecuencias_escolaridad_homologada.loc[
                        frecuencias_escolaridad_homologada.anio.eq(anio)
                        & frecuencias_escolaridad_homologada.nivel_educativo_homologado.notna(),
                        "frecuencia"
                    ].min()
                ),
            }.items()
        },
    }])
], ignore_index=True)
display(auditoria_escolaridad_por_anio)
display(frecuencias_escolaridad_homologada)
display(validaciones_escolaridad)
display(pd.DataFrame([{
    "estado": estado_escolaridad, "ampliado_preparado": EJECUTAR_AMPLIADO,
    "categorias_iguales": categorias_educativas_iguales,
    "referencia": "Secundaria",
}]))

## 3. Targets, diseño y propositos

Se conservan cuatro escalas: nivel nominal trimestral, log1p nominal, nivel real aproximado en pesos de 2024 y log1p real. El analisis **principal** usa log1p nominal. El nivel nominal es secundario; el log real aproximado permanece como sensibilidad **desactivada por defecto**. Los coeficientes de log1p se leen en la escala de 1+ingreso: para una dummy, 100(exp(beta)-1) es la diferencia porcentual en 1+ingreso frente a su referencia. No es exactamente una diferencia porcentual del ingreso. En particular, multiplicar un monto anual por un factor no equivale exactamente a sumar una constante tras log1p, aunque la discrepancia sea pequeña si el ingreso esta suficientemente alejado de cero.

El factor real actual fue calibrado con un benchmark de ingreso corriente **del hogar** y aplicado aqui al ingreso laboral **individual**. El target real no sera principal hasta aprobar un deflactor apropiado para ingreso laboral individual. Interceptos y montos nominales entre años no representan pesos de poder adquisitivo comparable. Los efectos fijos de año pooled absorben diferencias promedio condicionadas de nivel, pero no convierten ingresos nominales en reales ni solucionan por si solos toda la comparabilidad monetaria.

**Inferencia:** todas las personas validas, sin particion, con EE agrupados por hogar como criterio principal. HC3 es sensibilidad y los errores convencionales son referencia. Agrupar por hogar capta dependencia intrahogar, pero no reproduce estratos, UPM, ponderadores y replicas del diseño ENIGH. **Prediccion:** una eventual evaluacion separada usaria train/valid agrupados por hogar y preprocesamiento aprendido solo en train; MAE/RMSE no deciden significancia.

In [ ]:
variables_modelo_nucleo = (
    variables_comunes_numericas + variables_comunes_categoricas
    + variable_geografica_regional
)
variables_modelo_principal = (
    variables_comunes_numericas + variables_comunes_categoricas
    + ([VARIABLE_EDUCACION] if EJECUTAR_AMPLIADO else [])
    + variable_geografica_regional
)
ESPECIFICACIONES = {"regional_nucleo": variables_modelo_nucleo}
if EJECUTAR_AMPLIADO:
    ESPECIFICACIONES["regional_ampliado"] = variables_modelo_principal
if EJECUTAR_ESPECIFICACION_ENTIDAD:
    ESPECIFICACIONES["estatal_nucleo"] = (
        variables_comunes_numericas + variables_comunes_categoricas
        + variable_geografica_estatal
    )
    if EJECUTAR_AMPLIADO:
        ESPECIFICACIONES["estatal_ampliado_sensibilidad"] = (
            variables_comunes_numericas + variables_comunes_categoricas
            + [VARIABLE_EDUCACION] + variable_geografica_estatal
        )
ESPECIFICACION_PRINCIPAL = (
    "regional_ampliado" if EJECUTAR_AMPLIADO else "regional_nucleo"
)
if not HOMOLOGACION_APTA:
    print("Escolaridad bloqueada: nucleo preliminar; no es conclusion sustantiva definitiva.")
for nombre, variables in ESPECIFICACIONES.items():
    if "region_banxico" in variables and "entidad" in variables:
        raise ValueError(f"{nombre}: no combinar dos geografias.")
    if len(variables) != len(set(variables)):
        raise ValueError(f"{nombre}: predictores repetidos.")

ESCALA_PRINCIPAL = "log_nominal"
if TARGET_PRINCIPAL != "log_ingreso_nominal":
    raise ValueError("Target principal semantico inesperado.")
if nombres_targets(ANIO_REFERENCIA)[ESCALA_PRINCIPAL] != LOG_NOMINAL:
    raise ValueError("La escala principal no corresponde a la columna nominal existente.")
TARGETS_ANUALES = [ESCALA_PRINCIPAL, "nivel_nominal"]
if EJECUTAR_TARGET_REAL_APROX:
    TARGETS_ANUALES.append("log_real_2024")
display(pd.DataFrame([
    {"especificacion": nombre, "predictores": " | ".join(variables),
     "target_principal": TARGET_PRINCIPAL,
     "target_secundario": TARGET_NOMINAL,
     "target_real_aprox_activo": EJECUTAR_TARGET_REAL_APROX}
    for nombre, variables in ESPECIFICACIONES.items()
]))

def construir_matriz(base, variables):
    matriz = pd.DataFrame(index=base.index)
    info = [{"termino": "const", "variable": "intercepto",
             "categoria": None, "referencia": None, "tipo": "intercepto"}]
    for variable in variables:
        if variable in variables_comunes_numericas:
            serie = pd.to_numeric(base[variable], errors="coerce")
            if serie.isna().any() or not np.isfinite(serie).all():
                raise ValueError(f"{variable}: numerica no finita o faltante.")
            matriz[variable] = serie.astype(float)
            info.append({"termino": variable, "variable": variable,
                         "categoria": None, "referencia": None, "tipo": "numerica"})
        else:
            if variable not in CATEGORIAS_COMUNES:
                raise KeyError(f"Categorias no aprobadas para {variable}.")
            serie = base[variable].astype(str)
            if set(serie) != set(CATEGORIAS_COMUNES[variable]):
                raise ValueError(f"{variable}: categorias observadas distintas del contrato.")
            for categoria in CATEGORIAS_COMUNES[variable][1:]:
                termino = f"{variable}__{categoria}"
                matriz[termino] = serie.eq(categoria).astype(float)
                info.append({"termino": termino, "variable": variable,
                             "categoria": categoria,
                             "referencia": REFERENCIAS[variable],
                             "tipo": "categorica"})
    matriz = sm.add_constant(matriz, has_constant="add")
    if matriz.isna().any().any() or not np.isfinite(matriz.to_numpy()).all():
        raise ValueError("Matriz incompleta o no finita.")
    if np.linalg.matrix_rank(matriz.to_numpy().T @ matriz.to_numpy()) != matriz.shape[1]:
        raise ValueError("Matriz sin rango completo; revisar dependencia exacta.")
    return matriz, pd.DataFrame(info)


def codigos_hogar(base):
    codigos, _ = pd.factorize(base["id_hogar_anio"])
    if len(np.unique(codigos)) < 2:
        raise ValueError("Se requieren al menos dos hogares para EE agrupados.")
    return codigos


def variantes_covarianza(ajuste, base):
    return {
        "convencional": ajuste,
        "HC3": ajuste.get_robustcov_results(cov_type="HC3", use_t=False),
        "cluster_hogar": ajuste.get_robustcov_results(
            cov_type="cluster", groups=codigos_hogar(base),
            use_correction=True, use_t=False
        ),
    }


def coeficientes_de(modelo, info, contexto):
    meta = info.set_index("termino")
    terminos = modelo.model.exog_names
    intervalos = np.asarray(modelo.conf_int(alpha=ALFA))
    filas = []
    for j, termino in enumerate(terminos):
        detalle = meta.loc[termino]
        filas.append({
            **contexto, "termino": termino, "variable": detalle["variable"],
            "categoria": detalle["categoria"], "referencia": detalle["referencia"],
            "tipo_variable": detalle["tipo"],
            "coeficiente": float(np.asarray(modelo.params)[j]),
            "error_estandar": float(np.asarray(modelo.bse)[j]),
            "estadistico": float(np.asarray(modelo.tvalues)[j]),
            "p_value": float(np.asarray(modelo.pvalues)[j]),
            "intervalo_inferior": float(intervalos[j, 0]),
            "intervalo_superior": float(intervalos[j, 1]),
            "n_personas": int(modelo.nobs),
            "r2": float(modelo.rsquared),
            "r2_ajustado": float(modelo.rsquared_adj),
            "aic": float(modelo.aic), "bic": float(modelo.bic),
            "categoria_ausente": False, "modelo_no_estimable": False,
        })
    return pd.DataFrame(filas)


def pruebas_categoricas(modelo, info, contexto):
    filas = []
    nombres = modelo.model.exog_names
    for variable, bloque in info.loc[info.tipo.eq("categorica")].groupby("variable"):
        terminos = bloque.termino.tolist()
        restriccion = np.zeros((len(terminos), len(nombres)))
        for j, termino in enumerate(terminos):
            restriccion[j, nombres.index(termino)] = 1
        prueba = modelo.wald_test(restriccion, use_f=False, scalar=True)
        filas.append({
            **contexto, "variable": variable, "terminos_probados": " | ".join(terminos),
            "estadistico": float(prueba.statistic),
            "gl_restricciones": len(terminos), "gl_denominador": np.nan,
            "p_value": float(prueba.pvalue),
            "interpretacion": "H0: coeficientes del bloque iguales a cero; no causal.",
            "n_grupos": contexto["n_hogares"],
        })
    return pd.DataFrame(filas)

## 4. Modelos separados por año

Cada año recibe los mismos predictores, transformaciones y referencias aprobadas. OLS log nominal con escolaridad homologada es la especificacion sustantiva principal si la auditoria pasa; el nucleo sin escolaridad permite comparar betas sobre la misma muestra. El nivel nominal es secundario; el real aproximado solo se activa como sensibilidad. EE agrupados por hogar gobiernan conclusiones y Wald; HC3 es sensibilidad y los convencionales solo referencia del **mismo ajuste**. Un bloque categorico se contrasta conjuntamente. Un IC de 95% no implica causalidad.

La sensibilidad estatal reemplaza region; no se introducen ambas geografias en una regresion. La WLS usa factor normalizado y se ejecuta solo si factor, estrato y UPM pasan la auditoria. WLS y agrupacion por hogar **no** reconstruyen el diseño complejo completo.

In [ ]:
filas_metricas_anuales = []
tablas_coeficientes_anuales = []
tablas_pruebas_anuales = []
DISENO_VALIDO_CUATRO = bool(
    auditoria_bases_temporales.factor_valido_y_diseno_disponible.all()
)
PONDERADOS_EJECUTADOS = EJECUTAR_MODELOS_PONDERADOS and DISENO_VALIDO_CUATRO

for anio in ANIOS:
    base = bases_modelo[anio]
    targets = nombres_targets(anio)
    muestra_comun = base.loc[base[VARIABLE_EDUCACION].notna()] if EJECUTAR_AMPLIADO else base
    print(f"{anio}: perdida por escolaridad = {len(base) - len(muestra_comun)} personas")
    especificaciones_anio = dict(ESPECIFICACIONES)
    if EJECUTAR_AMPLIADO and len(muestra_comun) != len(base):
        especificaciones_anio["regional_nucleo_muestra_comun"] = variables_modelo_nucleo
    for especificacion, variables in especificaciones_anio.items():
        base_ajuste = (muestra_comun if ("ampliado" in especificacion or
                       especificacion.endswith("_muestra_comun")) else base)
        n_hogares = base_ajuste.id_hogar_anio.nunique()
        X, info = construir_matriz(base_ajuste, variables)
        for escala_target in TARGETS_ANUALES:
            y = pd.to_numeric(base_ajuste[targets[escala_target]], errors="coerce")
            if y.isna().any() or not np.isfinite(y).all():
                raise ValueError(f"{anio}: target {escala_target} no finito.")
            ajuste = sm.OLS(y, X).fit()
            if MOSTRAR_SUMMARIES_COMPLETOS and (
                especificacion == ESPECIFICACION_PRINCIPAL
                and escala_target == ESCALA_PRINCIPAL
            ):
                print(f"OLS principal {anio}: {especificacion}, {escala_target}; EE hogar")
                display(variantes_covarianza(ajuste, base_ajuste)[
                    COVARIANZA_INFERENCIA_PRINCIPAL
                ].summary())
            contexto = {
                "anio": anio, "especificacion": especificacion,
                "escala_target": escala_target, "estimador": "OLS",
                "n_hogares": n_hogares,
            }
            filas_metricas_anuales.append({
                **contexto, "n_personas": int(ajuste.nobs),
                "r2": float(ajuste.rsquared),
                "r2_ajustado": float(ajuste.rsquared_adj),
                "aic": float(ajuste.aic), "bic": float(ajuste.bic),
            })
            for covarianza, resultado in variantes_covarianza(ajuste, base_ajuste).items():
                etiqueta = {**contexto, "tipo_covarianza": covarianza}
                tablas_coeficientes_anuales.append(
                    coeficientes_de(resultado, info, etiqueta)
                )
                tablas_pruebas_anuales.append(
                    pruebas_categoricas(resultado, info, etiqueta)
                )
        if (
            PONDERADOS_EJECUTADOS and especificacion == ESPECIFICACION_PRINCIPAL
        ):
            pesos = pd.to_numeric(base_ajuste["factor"], errors="coerce")
            pesos_normalizados = pesos / pesos.mean()
            y = pd.to_numeric(base_ajuste[targets[ESCALA_PRINCIPAL]], errors="coerce")
            ajuste_wls = sm.WLS(y, X, weights=pesos_normalizados).fit()
            contexto = {
                "anio": anio, "especificacion": especificacion,
                "escala_target": ESCALA_PRINCIPAL, "estimador": "WLS_sensibilidad",
                "n_hogares": n_hogares,
            }
            filas_metricas_anuales.append({
                **contexto, "n_personas": int(ajuste_wls.nobs),
                "r2": float(ajuste_wls.rsquared),
                "r2_ajustado": float(ajuste_wls.rsquared_adj),
                "aic": float(ajuste_wls.aic), "bic": float(ajuste_wls.bic),
            })
            for covarianza, resultado in variantes_covarianza(ajuste_wls, base_ajuste).items():
                etiqueta = {**contexto, "tipo_covarianza": covarianza}
                tablas_coeficientes_anuales.append(
                    coeficientes_de(resultado, info, etiqueta)
                )
                tablas_pruebas_anuales.append(
                    pruebas_categoricas(resultado, info, etiqueta)
                )

metricas_modelos_anuales = pd.DataFrame(filas_metricas_anuales)
coeficientes_modelos_anuales = pd.concat(tablas_coeficientes_anuales, ignore_index=True)
pruebas_conjuntas_anuales = pd.concat(tablas_pruebas_anuales, ignore_index=True)
display(metricas_modelos_anuales)
display(pruebas_conjuntas_anuales.loc[
    pruebas_conjuntas_anuales.especificacion.eq(ESPECIFICACION_PRINCIPAL)
    & pruebas_conjuntas_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & pruebas_conjuntas_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & pruebas_conjuntas_anuales.estimador.eq("OLS")
])
display(pd.DataFrame([{
    "ponderados_solicitados": EJECUTAR_MODELOS_PONDERADOS,
    "diseno_disponible_cuatro_anios": DISENO_VALIDO_CUATRO,
    "ponderados_ejecutados": PONDERADOS_EJECUTADOS,
    "normalizacion": "factor dividido por su media dentro de cada año",
}]))

In [ ]:
orden_comparacion = [
    "estimador", "especificacion", "escala_target", "tipo_covarianza", "termino"
]
coeficientes_modelos_anuales = coeficientes_modelos_anuales.sort_values(
    orden_comparacion + ["anio"]
).reset_index(drop=True)
coeficientes_modelos_anuales["signo"] = np.sign(
    coeficientes_modelos_anuales.coeficiente
).astype(int)
coeficientes_modelos_anuales["significativa_descriptiva"] = (
    coeficientes_modelos_anuales.p_value < ALFA
)
grupos = coeficientes_modelos_anuales.groupby(orden_comparacion, dropna=False)
coeficientes_modelos_anuales["signo_previo"] = grupos.signo.shift()
coeficientes_modelos_anuales["significativa_previa"] = grupos.significativa_descriptiva.shift()
coeficientes_modelos_anuales["cambio_signo_descriptivo"] = (
    coeficientes_modelos_anuales.signo_previo.notna()
    & coeficientes_modelos_anuales.signo.ne(coeficientes_modelos_anuales.signo_previo)
)
coeficientes_modelos_anuales["entrada_significancia_descriptiva"] = (
    coeficientes_modelos_anuales.significativa_previa.eq(False)
    & coeficientes_modelos_anuales.significativa_descriptiva
)
coeficientes_modelos_anuales["salida_significancia_descriptiva"] = (
    coeficientes_modelos_anuales.significativa_previa.eq(True)
    & ~coeficientes_modelos_anuales.significativa_descriptiva
)
coeficientes_modelos_anuales["amplitud_intervalo"] = (
    coeficientes_modelos_anuales.intervalo_superior
    - coeficientes_modelos_anuales.intervalo_inferior
)
display(coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.especificacion.eq(ESPECIFICACION_PRINCIPAL)
    & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelos_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelos_anuales.estimador.eq("OLS"),
    ["anio", "variable", "termino", "coeficiente", "error_estandar",
     "intervalo_inferior", "intervalo_superior",
     "significativa_descriptiva", "cambio_signo_descriptivo"]
].head(40))

In [ ]:
filas_comparacion = []
if EJECUTAR_AMPLIADO:
    for anio in ANIOS:
        n_total = int(auditoria_bases_temporales.loc[
            auditoria_bases_temporales.anio.eq(anio), "n_personas"
        ].iloc[0])
        n_comun = int(bases_modelo[anio][VARIABLE_EDUCACION].notna().sum())
        etapas = [(
            "incorporacion_escolaridad_muestra_comun",
            "regional_nucleo_muestra_comun" if n_comun < n_total else "regional_nucleo",
            "regional_ampliado",
        )]
        if n_comun < n_total:
            etapas.insert(0, (
                "cambio_muestra_sin_escolaridad",
                "regional_nucleo", "regional_nucleo_muestra_comun"
            ))
        for etapa, nombre_nucleo, nombre_ampliado in etapas:
            filtro = (
                coeficientes_modelos_anuales.anio.eq(anio)
                & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
                & coeficientes_modelos_anuales.tipo_covarianza.eq(
                    COVARIANZA_INFERENCIA_PRINCIPAL
                )
                & coeficientes_modelos_anuales.estimador.eq("OLS")
            )
            anterior = coeficientes_modelos_anuales.loc[
                filtro & coeficientes_modelos_anuales.especificacion.eq(nombre_nucleo)
            ].set_index("termino")
            posterior = coeficientes_modelos_anuales.loc[
                filtro & coeficientes_modelos_anuales.especificacion.eq(nombre_ampliado)
            ].set_index("termino")
            if anterior.empty or posterior.empty:
                raise ValueError(f"{anio}: falta un modelo para comparar {etapa}.")
            for termino in anterior.index.intersection(posterior.index):
                a, b = anterior.loc[termino], posterior.loc[termino]
                delta = float(b.coeficiente - a.coeficiente)
                filas_comparacion.append({
                    "anio": anio, "etapa": etapa, "termino": termino,
                    "variable": a.variable, "modelo_nucleo": nombre_nucleo,
                    "modelo_ampliado": nombre_ampliado,
                    "tipo_covarianza": COVARIANZA_INFERENCIA_PRINCIPAL,
                    "n_total": n_total, "n_muestra_comun": n_comun,
                    "n_perdidas_educacion": n_total - n_comun,
                    "n_anterior": int(a.n_personas), "n_posterior": int(b.n_personas),
                    "coeficiente_nucleo": float(a.coeficiente),
                    "coeficiente_ampliado": float(b.coeficiente),
                    "cambio_absoluto": delta,
                    "cambio_porcentual": (
                        100 * delta / abs(float(a.coeficiente))
                        if abs(float(a.coeficiente)) > 1e-8 else np.nan
                    ),
                    "cambio_signo": int(np.sign(a.coeficiente) != np.sign(b.coeficiente)),
                    "ee_nucleo": float(a.error_estandar),
                    "ee_ampliado": float(b.error_estandar),
                    "cambio_ee": float(b.error_estandar - a.error_estandar),
                    "ic_nucleo_inferior": float(a.intervalo_inferior),
                    "ic_nucleo_superior": float(a.intervalo_superior),
                    "ic_ampliado_inferior": float(b.intervalo_inferior),
                    "ic_ampliado_superior": float(b.intervalo_superior),
                    "significativo_nucleo": bool(a.significativa_descriptiva),
                    "significativo_ampliado": bool(b.significativa_descriptiva),
                    "cambio_significancia": bool(
                        a.significativa_descriptiva != b.significativa_descriptiva
                    ),
                    "r2_nucleo": float(a.r2), "r2_ampliado": float(b.r2),
                    "cambio_r2": float(b.r2 - a.r2),
                    "r2_ajustado_nucleo": float(a.r2_ajustado),
                    "r2_ajustado_ampliado": float(b.r2_ajustado),
                    "cambio_r2_ajustado": float(b.r2_ajustado - a.r2_ajustado),
                    "aic_nucleo": float(a.aic), "aic_ampliado": float(b.aic),
                    "cambio_aic": float(b.aic - a.aic),
                    "bic_nucleo": float(a.bic), "bic_ampliado": float(b.bic),
                    "cambio_bic": float(b.bic - a.bic),
                    "interpretacion": "sensibilidad descriptiva; no prueba sesgo ni causalidad",
                })
comparacion_modelo_nucleo_ampliado = pd.DataFrame(filas_comparacion)
coeficientes_escolaridad_por_anio = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.especificacion.eq("regional_ampliado")
    & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.tipo_covarianza.eq(
        COVARIANZA_INFERENCIA_PRINCIPAL
    )
    & coeficientes_modelos_anuales.variable.eq(VARIABLE_EDUCACION)
].copy()
pruebas_conjuntas_escolaridad = pruebas_conjuntas_anuales.loc[
    pruebas_conjuntas_anuales.especificacion.eq("regional_ampliado")
    & pruebas_conjuntas_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & pruebas_conjuntas_anuales.estimador.eq("OLS")
    & pruebas_conjuntas_anuales.tipo_covarianza.eq(
        COVARIANZA_INFERENCIA_PRINCIPAL
    )
    & pruebas_conjuntas_anuales.variable.eq(VARIABLE_EDUCACION)
].copy()
display(comparacion_modelo_nucleo_ampliado.head(20))
display(coeficientes_escolaridad_por_anio)
display(pruebas_conjuntas_escolaridad)

## 5. Lectura visual de coeficientes anuales

Los forest plots muestran estimacion e IC de la especificacion principal log nominal con EE agrupados por hogar. Las trayectorias unen estimaciones de cortes distintos, no personas seguidas. Se separan variables numericas, binarias, laborales, escolaridad, localidad y geografia. La entidad se grafica en paneles legibles. Un cambio de signo o significancia anual es descriptivo; la prueba formal procede de interacciones con año.

In [ ]:
tabla_visual = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelos_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelos_anuales.variable.ne("intercepto")
].copy()
figuras_temporales = {}
colores_anio = {
    2018: "#2070a0", 2020: "#c55a35", 2022: "#488060", 2024: "#65527f"
}


def forest_por_terminos(tabla, terminos, titulo):
    sub = tabla.loc[tabla.termino.isin(terminos)]
    if sub.empty:
        raise ValueError(f"Sin terminos para {titulo}.")
    etiquetas = [t for t in terminos if t in set(sub.termino)]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.55 * len(etiquetas) + 1.5)))
    desplazamientos = {2018: -0.27, 2020: -0.09, 2022: 0.09, 2024: 0.27}
    posiciones = {termino: j for j, termino in enumerate(etiquetas)}
    for anio in ANIOS:
        corte = sub.loc[sub.anio.eq(anio)]
        for _, fila in corte.iterrows():
            y = posiciones[fila.termino] + desplazamientos[anio]
            ax.errorbar(
                fila.coeficiente, y,
                xerr=[[fila.coeficiente - fila.intervalo_inferior],
                      [fila.intervalo_superior - fila.coeficiente]],
                fmt="o", color=colores_anio[anio], capsize=2,
            )
        ax.plot([], [], "o", color=colores_anio[anio], label=str(anio))
    ax.axvline(0, color="0.35", linewidth=0.8)
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.invert_yaxis()
    ax.set_xlabel("Coeficiente log1p nominal e IC 95% (EE hogar)")
    ax.set_title(titulo)
    ax.legend(ncol=4, frameon=False)
    fig.tight_layout()
    return fig


def trayectorias_por_terminos(tabla, terminos, titulo):
    fig, ejes = plt.subplots(
        len(terminos), 1, figsize=(8, max(3, 2.3 * len(terminos))),
        squeeze=False, sharex=True,
    )
    for ax, termino in zip(ejes[:, 0], terminos):
        corte = tabla.loc[tabla.termino.eq(termino)].sort_values("anio")
        ax.errorbar(
            corte.anio, corte.coeficiente,
            yerr=[corte.coeficiente - corte.intervalo_inferior,
                  corte.intervalo_superior - corte.coeficiente],
            fmt="o-", capsize=3, color="#2070a0",
        )
        ax.axhline(0, color="0.5", linewidth=0.7)
        ax.set_ylabel(termino)
        ax.set_xticks(ANIOS)
    ejes[0, 0].set_title(titulo)
    ejes[-1, 0].set_xlabel("Corte ENIGH (personas diferentes)")
    fig.tight_layout()
    return fig


regional_visual = tabla_visual.loc[
    tabla_visual.especificacion.eq(ESPECIFICACION_PRINCIPAL)
]
grupos_figura = {
    "numericas": variables_comunes_numericas,
    "binarias": ["sexo_desc", "hablaind_desc", "segsoc_desc"],
    "laborales_categoricas": ["subor_principal_desc", "contrato_principal_desc"],
    "localidad": ["tam_loc_desc"],
    "region": ["region_banxico"],
    **({"escolaridad": [VARIABLE_EDUCACION]} if EJECUTAR_AMPLIADO else {}),
}
for nombre, variables in grupos_figura.items():
    terminos = regional_visual.loc[
        regional_visual.variable.isin(variables), "termino"
    ].drop_duplicates().tolist()
    fig = forest_por_terminos(regional_visual, terminos, nombre.replace("_", " ").title())
    figuras_temporales[f"forest_{nombre}"] = fig
    display(fig)
for nombre in ["numericas", "binarias"]:
    terminos = regional_visual.loc[
        regional_visual.variable.isin(grupos_figura[nombre]), "termino"
    ].drop_duplicates().tolist()
    fig = trayectorias_por_terminos(
        regional_visual, terminos, f"Trayectorias: {nombre}"
    )
    figuras_temporales[f"trayectorias_{nombre}"] = fig
    display(fig)

if EJECUTAR_ESPECIFICACION_ENTIDAD:
    estatal_visual = tabla_visual.loc[
        tabla_visual.especificacion.eq("estatal_ampliado_sensibilidad" if EJECUTAR_AMPLIADO else "estatal_nucleo")
        & tabla_visual.variable.eq("entidad")
    ]
    terminos_entidad = estatal_visual.termino.drop_duplicates().tolist()
    for inicio in range(0, len(terminos_entidad), 8):
        bloque = terminos_entidad[inicio:inicio + 8]
        fig = forest_por_terminos(
            estatal_visual, bloque,
            f"Entidades {inicio + 1}-{inicio + len(bloque)} (ref. Ciudad de Mexico)"
        )
        figuras_temporales[f"forest_entidad_{inicio // 8 + 1}"] = fig
        display(fig)

mapa = regional_visual.pivot(index="termino", columns="anio", values="coeficiente")
fig, ax = plt.subplots(figsize=(8, max(5, 0.42 * len(mapa) + 1)))
limite = float(np.nanmax(np.abs(mapa.to_numpy())))
sns.heatmap(mapa, cmap="BrBG", center=0, vmin=-limite, vmax=limite,
            annot=True, fmt=".2f", cbar_kws={"label": "beta log1p nominal"}, ax=ax)
ax.set_title("Signo y magnitud; no es prueba de cambio temporal")
ax.set_xlabel("Año")
ax.set_ylabel("Termino")
fig.tight_layout()
figuras_temporales["mapa_signos_magnitudes"] = fig
display(fig)

## 6. Modelo pooled con efectos de año

Se apilan cuatro cortes con identificadores explicitos de persona-año y hogar-año. La referencia temporal es 2024. Los coeficientes de año describen diferencias promedio condicionadas de **log1p nominal**; absorben diferencias medias de nivel, pero no deflactan los ingresos ni vuelven directamente comparables interceptos o montos nominales. La especificacion regional ampliada es principal si escolaridad pasa las validaciones; el nucleo es sensibilidad. La alternativa estatal reemplaza region. El clustering pooled usa hogar-año, nunca folios aislados. WLS divide factor por su media dentro de cada año y es sensibilidad, no inferencia de diseño muestral completo.

In [ ]:
columnas_pool = list(dict.fromkeys(
    KEY_COLS + ["id_hogar_anio", "id_persona_anio"]
    + variables_comunes_numericas + variables_comunes_categoricas
    + variable_geografica_regional + variable_geografica_estatal
    + [VARIABLE_EDUCACION, "factor", "est_dis", "upm"]
))
partes_pool = []
for anio in ANIOS:
    parte = bases_modelo[anio][columnas_pool].copy()
    nombres = nombres_targets(anio)
    for escala, columna in nombres.items():
        parte[f"target_{escala}"] = pd.to_numeric(
            bases_modelo[anio][columna], errors="coerce"
        ).to_numpy()
    partes_pool.append(parte)
apilada = pd.concat(partes_pool, ignore_index=True)
if len(apilada) != int(auditoria_bases_temporales.n_personas.sum()):
    raise ValueError("El apilado cambio el universo.")
if apilada.id_persona_anio.duplicated().any() or apilada.duplicated(KEY_COLS).any():
    raise ValueError("Persona-año repetida al apilar cortes.")
if apilada.groupby("id_hogar_anio").anio.nunique().gt(1).any():
    raise ValueError("Colision de hogar-año entre cortes.")
if apilada.groupby("anio").id_hogar_anio.nunique().to_dict() != dict(zip(
    ANIOS, auditoria_bases_temporales.n_hogares.tolist()
)):
    raise ValueError("Conteos de hogares-año alterados al apilar.")
anios_otros = [anio for anio in ANIOS if anio != ANIO_REFERENCIA]
apilada_principal = (
    apilada.loc[apilada[VARIABLE_EDUCACION].notna()].copy()
    if EJECUTAR_AMPLIADO else apilada
)
n_hogares_pool = apilada_principal.id_hogar_anio.nunique()


def matriz_pooled(base, variables):
    X, info = construir_matriz(base, variables)
    for anio in anios_otros:
        termino = f"anio__{anio}"
        X[termino] = base.anio.eq(anio).astype(float)
        info = pd.concat([
            info, pd.DataFrame([{
                "termino": termino, "variable": "anio",
                "categoria": anio, "referencia": ANIO_REFERENCIA,
                "tipo": "efecto_anio",
            }])
        ], ignore_index=True)
    if np.linalg.matrix_rank(X.to_numpy().T @ X.to_numpy()) != X.shape[1]:
        raise ValueError("Pooled sin rango completo.")
    return X, info


filas_coeficientes_pooled = []
resumen_pooled = []
especificaciones_pool = {
    "pooled_regional_nucleo": (apilada, variables_modelo_nucleo),
}
if EJECUTAR_AMPLIADO:
    especificaciones_pool["pooled_regional_ampliado"] = (
        apilada_principal, variables_modelo_principal
    )
    if len(apilada_principal) != len(apilada):
        especificaciones_pool["pooled_regional_nucleo_muestra_comun"] = (
            apilada_principal, variables_modelo_nucleo
        )
if EJECUTAR_ESPECIFICACION_ENTIDAD:
    nombre_estatal = (
        "estatal_ampliado_sensibilidad" if EJECUTAR_AMPLIADO else "estatal_nucleo"
    )
    especificaciones_pool["pooled_" + nombre_estatal] = (
        apilada_principal, ESPECIFICACIONES[nombre_estatal]
    )
ESPECIFICACION_POOLED_PRINCIPAL = (
    "pooled_regional_ampliado" if EJECUTAR_AMPLIADO else "pooled_regional_nucleo"
)
for nombre, (base_ajuste, variables) in especificaciones_pool.items():
    X, info = matriz_pooled(base_ajuste, variables)
    y = base_ajuste["target_" + ESCALA_PRINCIPAL].astype(float)
    ajuste = sm.OLS(y, X).fit()
    if nombre == ESPECIFICACION_POOLED_PRINCIPAL:
        X_pooled_regional, info_pooled_regional = X, info
        y_pool = y
        ajuste_pooled_regional = ajuste
        apilada_interacciones = base_ajuste
        if MOSTRAR_SUMMARIES_COMPLETOS:
            display(variantes_covarianza(ajuste, base_ajuste)[
                COVARIANZA_INFERENCIA_PRINCIPAL
            ].summary())
    for covarianza, resultado in variantes_covarianza(ajuste, base_ajuste).items():
        filas_coeficientes_pooled.append(coeficientes_de(
            resultado, info,
            {"especificacion": nombre, "escala_target": ESCALA_PRINCIPAL,
             "estimador": "OLS", "tipo_covarianza": covarianza,
             "n_hogares": base_ajuste.id_hogar_anio.nunique()}
        ))
    resumen_pooled.append({
        "especificacion": nombre, "estimador": "OLS",
        "n_personas": len(base_ajuste),
        "n_hogares": base_ajuste.id_hogar_anio.nunique(),
        "r2": ajuste.rsquared, "r2_ajustado": ajuste.rsquared_adj,
        "aic": ajuste.aic, "bic": ajuste.bic,
    })
if EJECUTAR_TARGET_REAL_APROX:
    ajuste_real = sm.OLS(
        apilada_interacciones["target_log_real_2024"].astype(float),
        X_pooled_regional
    ).fit()
    for covarianza, resultado in variantes_covarianza(
        ajuste_real, apilada_interacciones
    ).items():
        filas_coeficientes_pooled.append(coeficientes_de(
            resultado, info_pooled_regional,
            {"especificacion": ESPECIFICACION_POOLED_PRINCIPAL,
             "escala_target": "log_real_2024_sensibilidad",
             "estimador": "OLS", "tipo_covarianza": covarianza,
             "n_hogares": n_hogares_pool}
        ))
if PONDERADOS_EJECUTADOS:
    pesos_pool = pd.to_numeric(apilada_interacciones.factor, errors="coerce")
    pesos_pool = pesos_pool / pesos_pool.groupby(
        apilada_interacciones.anio
    ).transform("mean")
    ajuste_pooled_wls = sm.WLS(
        y_pool, X_pooled_regional, weights=pesos_pool
    ).fit()
    for covarianza, resultado in variantes_covarianza(
        ajuste_pooled_wls, apilada_interacciones
    ).items():
        filas_coeficientes_pooled.append(coeficientes_de(
            resultado, info_pooled_regional,
            {"especificacion": ESPECIFICACION_POOLED_PRINCIPAL,
             "escala_target": ESCALA_PRINCIPAL, "estimador": "WLS_sensibilidad",
             "tipo_covarianza": covarianza, "n_hogares": n_hogares_pool}
        ))
    resumen_pooled.append({
        "especificacion": ESPECIFICACION_POOLED_PRINCIPAL,
        "estimador": "WLS_sensibilidad",
        "n_personas": len(apilada_interacciones),
        "n_hogares": n_hogares_pool,
        "r2": ajuste_pooled_wls.rsquared,
        "r2_ajustado": ajuste_pooled_wls.rsquared_adj,
        "aic": ajuste_pooled_wls.aic, "bic": ajuste_pooled_wls.bic,
    })
coeficientes_modelo_pooled = pd.concat(filas_coeficientes_pooled, ignore_index=True)
display(pd.DataFrame(resumen_pooled))
display(coeficientes_modelo_pooled.loc[
    coeficientes_modelo_pooled.variable.eq("anio")
    & coeficientes_modelo_pooled.especificacion.eq(ESPECIFICACION_POOLED_PRINCIPAL)
    & coeficientes_modelo_pooled.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelo_pooled.estimador.eq("OLS"),
    ["especificacion", "termino", "coeficiente", "intervalo_inferior",
     "intervalo_superior", "p_value"]
])

## 7. Interacciones con año

El coeficiente principal de un predictor es su asociacion en **2024**. La interaccion predictor × 2018, 2020 o 2022 es la diferencia condicionada frente a 2024. El efecto total en otro año suma coeficiente principal e interaccion; su IC usa la covarianza del mismo resultado agrupado por hogar-año. Para escolaridad, cada dummy y efecto total se lee respecto de **Secundaria**, no como rendimiento causal de obtener un titulo. Una prima educativa observada puede reflejar experiencia potencial, calidad educativa, campo de estudio u otros factores no disponibles. No se infiere cambio a partir de dos p-values anuales distintos.

Por defecto se incluyen interacciones de todos los bloques del modelo regional principal, incluida escolaridad cuando este validada. La prueba conjunta del bloque educativo contrasta todas sus interacciones con año. Una ejecucion acotada puede especificar bloques manuales, pero omitir escolaridad deja sin estimar su contraste temporal y debe quedar etiquetado. La alternativa estatal permanece separada.

In [ ]:
if EJECUTAR_INTERACCIONES_COMPLETAS:
    variables_interaccion = (
        BLOQUES["individuales"] + BLOQUES["laborales"]
        + BLOQUES["localidad"] + BLOQUES["region"]
        + (BLOQUES["escolaridad"] if EJECUTAR_AMPLIADO else [])
    )
else:
    bloques_invalidos = sorted(set(BLOQUES_INTERACCION) - set(BLOQUES))
    if bloques_invalidos:
        raise ValueError(f"Bloques de interaccion desconocidos: {bloques_invalidos}")
    variables_interaccion = list(dict.fromkeys(
        [v for bloque in BLOQUES_INTERACCION for v in BLOQUES[bloque]]
        + VARIABLES_INTERACCION_MANUAL
    ))
variables_interaccion = list(dict.fromkeys(variables_interaccion))
if not variables_interaccion or not set(variables_interaccion).issubset(
    ESPECIFICACIONES[ESPECIFICACION_PRINCIPAL]
):
    raise ValueError("Interacciones vacias o fuera de la especificacion regional.")
display(pd.DataFrame([
    {"bloque": bloque, "variables_interactuadas": " | ".join(
        v for v in lista if v in variables_interaccion
    ) or "no incluidas"}
    for bloque, lista in BLOQUES.items()
]))

X_interacciones = X_pooled_regional.copy()
info_interacciones = info_pooled_regional.copy()
filas_interacciones = []
info_base = info_pooled_regional.loc[
    info_pooled_regional.variable.isin(variables_interaccion)
    & ~info_pooled_regional.tipo.eq("efecto_anio")
].copy()
for variable in variables_interaccion:
    terminos_base = info_base.loc[info_base.variable.eq(variable)]
    if terminos_base.empty:
        raise ValueError(f"Sin terminos base para {variable}.")
    for _, detalle in terminos_base.iterrows():
        for anio in anios_otros:
            termino = f"{detalle.termino}:anio__{anio}"
            X_interacciones[termino] = (
                X_pooled_regional[detalle.termino]
                * X_pooled_regional[f"anio__{anio}"]
            )
            registro = {
                "termino": termino, "variable": variable,
                "categoria": detalle.categoria,
                "referencia": detalle.referencia, "tipo": "interaccion",
                "termino_base": detalle.termino, "anio": anio,
            }
            filas_interacciones.append(registro)
            info_interacciones = pd.concat([
                info_interacciones,
                pd.DataFrame([{k: registro[k] for k in [
                    "termino", "variable", "categoria", "referencia", "tipo"
                ]}])
            ], ignore_index=True)
interacciones_en_memoria = pd.DataFrame(filas_interacciones)
if np.linalg.matrix_rank(
    X_interacciones.to_numpy().T @ X_interacciones.to_numpy()
) != X_interacciones.shape[1]:
    raise ValueError("Interacciones sin rango completo; revisar categorias.")
ajuste_interacciones = sm.OLS(y_pool, X_interacciones).fit()
modelo_interacciones_principal = variantes_covarianza(
    ajuste_interacciones, apilada_interacciones
)[COVARIANZA_INFERENCIA_PRINCIPAL]
if MOSTRAR_SUMMARIES_COMPLETOS:
    display(modelo_interacciones_principal.summary())
coeficientes_modelo_pooled = pd.concat([
    coeficientes_modelo_pooled,
    coeficientes_de(
        modelo_interacciones_principal, info_interacciones,
        {"especificacion": "pooled_interacciones_regionales",
         "escala_target": ESCALA_PRINCIPAL, "estimador": "OLS",
         "tipo_covarianza": COVARIANZA_INFERENCIA_PRINCIPAL, "n_hogares": n_hogares_pool}
    )
], ignore_index=True)
display(interacciones_en_memoria[
    ["variable", "termino_base", "anio", "termino"]
].head(20))

## 8. Efectos totales y pruebas de Wald

Se calculan efectos totales por corte e IC de combinaciones lineales usando coeficientes y covarianza **del mismo ajuste agrupado por hogar-año**. Los Wald contrastan interacciones individuales, bloques categoricos (incluida escolaridad), familias y conjunto total. La tabla registra covarianza, terminos, estadistico, grados de libertad asintoticos, p-value y numero de grupos. P pequeño indica evidencia de diferencia condicionada, no causalidad. Una categoria educativa significativa en un año y no en otro no demuestra cambio; se consulta su interaccion y el Wald conjunto educativo. Multiplicidad queda pendiente.

In [ ]:
nombres_inter = modelo_interacciones_principal.model.exog_names
parametros_inter = np.asarray(modelo_interacciones_principal.params)
cov_inter = np.asarray(modelo_interacciones_principal.cov_params())
zcrit = norm.ppf(1 - ALFA / 2)


def contraste_lineal(pesos):
    beta = float(pesos @ parametros_inter)
    varianza = float(pesos @ cov_inter @ pesos)
    if varianza < -1e-10:
        raise ValueError("Varianza negativa del contraste.")
    ee = float(np.sqrt(max(varianza, 0)))
    z = beta / ee if ee > 0 else np.nan
    return {
        "coeficiente": beta, "error_estandar": ee,
        "intervalo_inferior": beta - zcrit * ee,
        "intervalo_superior": beta + zcrit * ee,
        "p_value": float(2 * norm.sf(abs(z))) if np.isfinite(z) else np.nan,
    }


filas_efectos = []
for termino_base, bloque in interacciones_en_memoria.groupby("termino_base", sort=False):
    detalle = bloque.iloc[0]
    for anio in ANIOS:
        pesos_total = np.zeros(len(nombres_inter))
        pesos_total[nombres_inter.index(termino_base)] = 1
        pesos_diferencia = np.zeros(len(nombres_inter))
        if anio != ANIO_REFERENCIA:
            termino = bloque.loc[bloque.anio.eq(anio), "termino"].iloc[0]
            pesos_total[nombres_inter.index(termino)] = 1
            pesos_diferencia[nombres_inter.index(termino)] = 1
        total = contraste_lineal(pesos_total)
        diferencia = (
            contraste_lineal(pesos_diferencia) if anio != ANIO_REFERENCIA
            else {"coeficiente": 0.0, "error_estandar": 0.0,
                  "intervalo_inferior": 0.0, "intervalo_superior": 0.0,
                  "p_value": np.nan}
        )
        filas_efectos.append({
            "anio": anio, "anio_referencia": ANIO_REFERENCIA,
            "variable": detalle.variable, "termino_base": termino_base,
            "categoria": detalle.categoria, "referencia": detalle.referencia,
            **{f"efecto_total_{k}": v for k, v in total.items()},
            **{f"diferencia_vs_2024_{k}": v for k, v in diferencia.items()},
            "tipo_covarianza": COVARIANZA_INFERENCIA_PRINCIPAL,
            "n_personas": len(apilada_interacciones), "n_hogares": n_hogares_pool,
            "n_grupos": n_hogares_pool,
        })
efectos_totales_interacciones = pd.DataFrame(filas_efectos)
display(efectos_totales_interacciones.head(20))


def prueba_wald_temporal(nombre, variables, tipo):
    terminos = interacciones_en_memoria.loc[
        interacciones_en_memoria.variable.isin(variables), "termino"
    ].tolist()
    if not terminos:
        return {
            "hipotesis": nombre, "tipo": tipo, "variables": " | ".join(variables),
            "terminos_probados": "", "estadistico": np.nan,
            "gl_restricciones": 0, "gl_denominador": np.nan, "p_value": np.nan,
            "interpretacion": "No incluido en la ejecucion parcial.",
            "conclusion_descriptiva": "no_estimado",
            "tipo_covarianza": COVARIANZA_INFERENCIA_PRINCIPAL,
            "n_grupos": n_hogares_pool,
        }
    R = np.zeros((len(terminos), len(nombres_inter)))
    for j, termino in enumerate(terminos):
        R[j, nombres_inter.index(termino)] = 1
    prueba = modelo_interacciones_principal.wald_test(
        R, use_f=False, scalar=True
    )
    p = float(prueba.pvalue)
    return {
        "hipotesis": nombre, "tipo": tipo,
        "variables": " | ".join(variables),
        "terminos_probados": " | ".join(terminos),
        "estadistico": float(prueba.statistic),
        "gl_restricciones": len(terminos),
        "gl_denominador": np.nan, "p_value": p,
        "tipo_covarianza": COVARIANZA_INFERENCIA_PRINCIPAL,
        "n_grupos": n_hogares_pool,
        "interpretacion": "H0: interacciones de estos terminos iguales a cero.",
        "conclusion_descriptiva": (
            "evidencia de diferencia temporal condicionada, no causal"
            if p < ALFA else
            "sin evidencia suficiente de diferencia temporal condicionada"
        ),
    }


filas_wald = []
for variable in variables_comunes_numericas:
    filas_wald.append(
        prueba_wald_temporal(variable, [variable], "continua_por_anio")
    )
for variable in variables_comunes_categoricas + ["region_banxico"] + (
    [VARIABLE_EDUCACION] if EJECUTAR_AMPLIADO else []
):
    filas_wald.append(
        prueba_wald_temporal(variable, [variable], "bloque_categorico_por_anio")
    )
for bloque, variables in {
    "individuales": BLOQUES["individuales"],
    "laborales": BLOQUES["laborales"],
    "territoriales": BLOQUES["localidad"] + BLOQUES["region"],
    **({"escolaridad": BLOQUES["escolaridad"]} if EJECUTAR_AMPLIADO else {}),
    "todas_las_interacciones": variables_interaccion,
}.items():
    filas_wald.append(
        prueba_wald_temporal(bloque, variables, "familia_temporal")
    )
pruebas_wald_temporales = pd.DataFrame(filas_wald)
display(pruebas_wald_temporales[
    ["hipotesis", "tipo", "tipo_covarianza", "n_grupos",
     "gl_restricciones", "estadistico", "p_value",
     "conclusion_descriptiva"]
])
interacciones_escolaridad_anio = pd.concat([
    coeficientes_modelo_pooled.loc[
        coeficientes_modelo_pooled.especificacion.eq("pooled_interacciones_regionales")
        & coeficientes_modelo_pooled.variable.eq(VARIABLE_EDUCACION)
        & coeficientes_modelo_pooled.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    ].assign(origen="coeficiente_modelo"),
    efectos_totales_interacciones.loc[
        efectos_totales_interacciones.variable.eq(VARIABLE_EDUCACION)
    ].assign(origen="efecto_total_por_anio"),
], ignore_index=True)
display(interacciones_escolaridad_anio)

## 9. Geografia, ponderacion y antecedente 2024

Los coeficientes regionales/estatales comparan el **intercepto condicionado** con Centro/Ciudad de Mexico, respectivamente; no dicen que todas las pendientes cambien por territorio. Para estudiar pendientes heterogeneas por estado haria falta una fase posterior, inicialmente con interacciones regionales justificadas. En este notebook las interacciones territoriales son con **año**, no con todos los predictores de cada entidad.

Factor, est_dis y upm se auditan. La WLS normaliza factor a media uno dentro de cada corte, tanto en modelos anuales como pooled; asi se controla la escala numerica y se evita que la suma de factores de un año domine automaticamente el pooled. La sensibilidad sigue sin tratar formalmente estratificacion, UPM y replicas del diseño ENIGH. Agrupar EE por hogar atiende dependencia intrahogar, no sustituye dicho diseño.

La lista historica de variables significativas de 2024 es solo un **antecedente descriptivo** del notebook 12/13 y no filtra el nucleo. La tabla siguiente reconstruye significancia por termino de la **nueva especificacion** cuando se ejecute, por lo que no debe confundirse con los coeficientes de entrenamiento del notebook 13.

In [ ]:
comparacion_geografica = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelos_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelos_anuales.especificacion.isin([
        ESPECIFICACION_PRINCIPAL,
        "estatal_ampliado_sensibilidad" if EJECUTAR_AMPLIADO else "estatal_nucleo",
    ])
    & coeficientes_modelos_anuales.variable.isin(["region_banxico", "entidad"])
].copy()
comparacion_geografica["origen_modelo"] = "anual"
geo_pooled = coeficientes_modelo_pooled.loc[
    coeficientes_modelo_pooled.estimador.eq("OLS")
    & coeficientes_modelo_pooled.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelo_pooled.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelo_pooled.especificacion.isin([
        ESPECIFICACION_POOLED_PRINCIPAL,
        "pooled_estatal_ampliado_sensibilidad" if EJECUTAR_AMPLIADO
        else "pooled_estatal_nucleo",
    ])
    & coeficientes_modelo_pooled.variable.isin(["region_banxico", "entidad"])
    & ~coeficientes_modelo_pooled.especificacion.eq(
        "pooled_interacciones_regionales"
    )
].copy()
geo_pooled["anio"] = pd.NA
geo_pooled["origen_modelo"] = "pooled"
comparacion_geografica = pd.concat(
    [comparacion_geografica, geo_pooled], ignore_index=True
)
comparacion_geografica["interpretacion"] = (
    "diferencia de intercepto condicionado vs referencia geografica; "
    "no implica cambio de pendientes ni causalidad"
)
display(comparacion_geografica[
    ["origen_modelo", "anio", "especificacion", "variable", "categoria",
     "referencia", "coeficiente", "intervalo_inferior", "intervalo_superior"]
].head(40))

ANTECEDENTE_2024_VARIABLES = [
    "edad", "n_trabajos", "horas_trabajos_total", "sexo_desc",
    "nivelaprob_desc", "parentesco_desc", "hablaind_desc",
    "segsoc_desc", "subor_principal_desc", "contrato_principal_desc",
    "region_banxico", "tam_loc_desc",
]
principal_anual = coeficientes_modelos_anuales.loc[
    coeficientes_modelos_anuales.estimador.eq("OLS")
    & coeficientes_modelos_anuales.especificacion.eq(ESPECIFICACION_PRINCIPAL)
    & coeficientes_modelos_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & coeficientes_modelos_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
    & coeficientes_modelos_anuales.variable.ne("intercepto")
].copy()
wald_por_variable = pruebas_wald_temporales.loc[
    pruebas_wald_temporales.tipo.isin(
        ["continua_por_anio", "bloque_categorico_por_anio"]
    )
].set_index("hipotesis")
prueba_2024 = pruebas_conjuntas_anuales.loc[
    pruebas_conjuntas_anuales.anio.eq(2024)
    & pruebas_conjuntas_anuales.estimador.eq("OLS")
    & pruebas_conjuntas_anuales.especificacion.eq(ESPECIFICACION_PRINCIPAL)
    & pruebas_conjuntas_anuales.escala_target.eq(ESCALA_PRINCIPAL)
    & pruebas_conjuntas_anuales.tipo_covarianza.eq(COVARIANZA_INFERENCIA_PRINCIPAL)
].set_index("variable")
filas_antecedentes = []
for _, fila in principal_anual.loc[principal_anual.anio.eq(2024)].iterrows():
    otros = principal_anual.loc[
        principal_anual.termino.eq(fila.termino)
        & principal_anual.anio.ne(2024)
    ].sort_values("anio")
    p_wald = (
        float(wald_por_variable.loc[fila.variable, "p_value"])
        if fila.variable in wald_por_variable.index else np.nan
    )
    p_bloque = (
        float(prueba_2024.loc[fila.variable, "p_value"])
        if fila.variable in prueba_2024.index else np.nan
    )
    filas_antecedentes.append({
        "termino": fila.termino, "variable": fila.variable,
        "en_antecedente_exploratorio_2024":
            fila.variable in ANTECEDENTE_2024_VARIABLES
            or (fila.variable == VARIABLE_EDUCACION and "nivelaprob_desc" in ANTECEDENTE_2024_VARIABLES),
        "significativa_2024_principal_actual":
            bool(fila.significativa_descriptiva),
        "signo_2024": int(fila.signo),
        "otros_anios_descriptivo": " | ".join(
            f"{int(r.anio)}: signo {int(r.signo)}, "
            f"p<{ALFA}: {bool(r.significativa_descriptiva)}"
            for _, r in otros.iterrows()
        ),
        "p_wald_temporal_variable": p_wald,
        "interaccion_temporal_significativa":
            bool(p_wald < ALFA) if np.isfinite(p_wald) else None,
        "p_prueba_conjunta_2024": p_bloque,
        "conclusion": (
            "diferencia temporal condicionada; no causal"
            if np.isfinite(p_wald) and p_wald < ALFA else
            "cambio de p anual no demuestra diferencia temporal"
        ),
    })
for variable in (["parentesco_desc"] + (["nivelaprob_desc"] if not EJECUTAR_AMPLIADO else [])):
    filas_antecedentes.append({
        "termino": None, "variable": variable,
        "en_antecedente_exploratorio_2024": True,
        "significativa_2024_principal_actual": None,
        "signo_2024": None, "otros_anios_descriptivo": None,
        "p_wald_temporal_variable": np.nan,
        "interaccion_temporal_significativa": None,
        "p_prueba_conjunta_2024": np.nan,
        "conclusion": "pendiente de armonizacion; fuera del modelo principal",
    })
antecedentes_2024 = pd.DataFrame(filas_antecedentes)
display(antecedentes_2024)

## 10. Guia de interpretacion y limites

- **Significativa en ambos años, betas diferentes:** las dos asociaciones pueden ser distintas; solo el contraste de interaccion y su IC evalúan formalmente esa diferencia.
- **Significativa solo en uno:** la diferencia entre estimaciones puede seguir siendo compatible con cero.
- **Dummies educativas:** cada coeficiente compara su nivel homologado con Secundaria, condicionado a las demas variables. El nivel original se conserva; el grupo homologado sacrifica detalle para comparar años. La prima observada no es efecto causal.
- **Bloque significativo:** un Wald conjunto responde otra pregunta que el p-value de una categoria.
- **Magnitud sustantiva:** para log1p, 100(exp(beta)-1) aplica a 1+ingreso; no confundirlo con porcentaje exacto del ingreso.
- **Nivel monetario:** no equiparar pesos nominales de años distintos; el deflactor individual requiere aprobacion.
- **Alcance:** solo personas adultas con ingreso positivo; sin seguimiento longitudinal ni control completo por experiencia potencial, calidad educativa, campo de estudio o diseño muestral complejo.
- **Pendiente:** revision manual de homologacion, pruebas multiples, pertinencia del factor real individual e inferencia con estratos/UPM/replicas.

## 11. Exportacion al final

Todas las tablas y figuras provienen de objetos creados durante **esta misma ejecucion manual**. El manifest registra la homologacion preparada y el estado pendiente de revision humana; esta edicion estatica no produjo resultados. No se leen coeficientes ni graficas anteriores.

In [ ]:
tablas_salida = {
    "auditoria_bases_temporales": auditoria_bases_temporales,
    "comparabilidad_variables": comparabilidad_variables,
    "referencias_categoricas": referencias_categoricas,
    "auditoria_escolaridad_por_anio": auditoria_escolaridad_por_anio,
    "mapeo_escolaridad_homologada": mapeo_escolaridad_homologada,
    "frecuencias_escolaridad_homologada": frecuencias_escolaridad_homologada,
    "comparacion_modelo_nucleo_ampliado": comparacion_modelo_nucleo_ampliado,
    "coeficientes_escolaridad_por_anio": coeficientes_escolaridad_por_anio,
    "pruebas_conjuntas_escolaridad": pruebas_conjuntas_escolaridad,
    "interacciones_escolaridad_anio": interacciones_escolaridad_anio,
    "validaciones_escolaridad": validaciones_escolaridad,
    "metricas_modelos_anuales": metricas_modelos_anuales,
    "coeficientes_modelos_anuales": coeficientes_modelos_anuales,
    "pruebas_conjuntas_anuales": pruebas_conjuntas_anuales,
    "coeficientes_modelo_pooled": coeficientes_modelo_pooled,
    "efectos_totales_interacciones": efectos_totales_interacciones,
    "pruebas_wald_temporales": pruebas_wald_temporales,
    "comparacion_geografica": comparacion_geografica,
    "antecedentes_2024": antecedentes_2024,
}
manifest_comparacion_temporal = {
    "estado": "pendiente_ejecucion_manual",
    "nota_estado": "Notebook preparado; resultados requieren ejecucion y revision humana.",
    "unidad_analisis": "persona",
    "universo": "edad>=18 e ingreso laboral/de negocio individual nominal trimestral>0",
    "anios": ANIOS,
    "anio_referencia": ANIO_REFERENCIA,
    "archivos_entrada": {
        str(anio): str(ruta.relative_to(PROJECT_ROOT))
        for anio, ruta in INPUTS.items()
    },
    "variables_comunes_numericas": variables_comunes_numericas,
    "variables_comunes_categoricas": variables_comunes_categoricas,
    "geografia_principal": variable_geografica_regional,
    "geografia_sensibilidad": variable_geografica_estatal
                              if EJECUTAR_ESPECIFICACION_ENTIDAD else [],
    "variables_ampliadas": [VARIABLE_EDUCACION] if EJECUTAR_AMPLIADO else [],
    "variable_escolaridad_original": "nivelaprob_desc",
    "variable_escolaridad_homologada": VARIABLE_EDUCACION,
    "codigo_educativo_en_base": False,
    "mapeo_escolaridad": mapeo_escolaridad_homologada.to_dict(orient="records"),
    "validaciones_escolaridad": validaciones_escolaridad.to_dict(orient="records"),
    "categorias_escolaridad": CATEGORIAS_COMUNES.get(VARIABLE_EDUCACION, []),
    "referencia_escolaridad": REFERENCIAS[VARIABLE_EDUCACION],
    "anios_escolaridad": ANIOS,
    "modelo_principal": ESPECIFICACION_PRINCIPAL if EJECUTAR_AMPLIADO else "nucleo_preliminar",
    "modelo_nucleo_sin_escolaridad": "regional_nucleo",
    "modelo_ampliado_con_escolaridad": "regional_ampliado" if EJECUTAR_AMPLIADO else None,
    "estado_escolaridad": estado_escolaridad,
    "variables_pendientes": variables_pendientes,
    "referencias": REFERENCIAS,
    "categorias_comunes": CATEGORIAS_COMUNES,
    "targets": {
        str(anio): nombres_targets(anio) for anio in ANIOS
    },
    "target_modelo_principal": LOG_NOMINAL,
    "target_principal_semantico": TARGET_PRINCIPAL,
    "target_real_aprox_es_sensibilidad": TARGET_REAL_APROX_ES_SENSIBILIDAD,
    "target_real_aprox_activado": EJECUTAR_TARGET_REAL_APROX,
    "target_modelo_secundario": "nivel_nominal",
    "estimadores": ["OLS"] + (
        ["WLS_sensibilidad"] if PONDERADOS_EJECUTADOS else []
    ),
    "covarianzas_anuales": ["convencional", "HC3", "cluster_hogar"],
    "covarianza_inferencia_principal": COVARIANZA_INFERENCIA_PRINCIPAL,
    "covarianza_sensibilidad": COVARIANZA_SENSIBILIDAD,
    "covarianza_interacciones": COVARIANZA_INFERENCIA_PRINCIPAL,
    "grupo_cluster_pooled": "id_hogar_anio",
    "interacciones": variables_interaccion,
    "interacciones_completas": EJECUTAR_INTERACCIONES_COMPLETAS,
    "ponderacion_sensibilidad_ejecutada": PONDERADOS_EJECUTADOS,
    "normalizacion_factor": "factor/media(factor) dentro de cada año",
    "diseno": "factor, est_dis, upm disponibles; WLS+EE hogar no es diseño complejo completo",
    "advertencia_monetaria": (
        "Factor anual calibrado al benchmark de ingreso corriente del hogar; "
        "uso en ingreso laboral individual pendiente de aprobacion metodologica."
    ),
    "limitaciones": [
        "asociaciones condicionadas, no efectos causales",
        "categorias educativas amplias pierden detalle; Especialidad solo en 2024",
        "experiencia potencial, calidad educativa y campo de estudio no controlados",
        "montos/interceptos nominales no son pesos reales comparables",
        "efectos fijos de año no sustituyen un deflactor individual aprobado",
        "log1p(factor*ingreso) no equivale exactamente a log1p(ingreso)+log(factor)",
        "EE hogar no reproduce diseño ENIGH completo",
    ],
    "metodo": "asociaciones condicionadas en cortes transversales repetidos, no causales",
    "versiones": {
        "pandas": pd.__version__, "numpy": np.__version__,
        "statsmodels": sm.__version__,
    },
}
if EXPORTAR_RESULTADOS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for nombre, tabla in tablas_salida.items():
        tabla.to_csv(OUTPUT_DIR / f"{nombre}.csv", index=False)
    figuras_dir = OUTPUT_DIR / "figures"
    figuras_dir.mkdir(parents=True, exist_ok=True)
    for nombre, figura in figuras_temporales.items():
        figura.savefig(figuras_dir / f"{nombre}.png", dpi=150,
                       bbox_inches="tight")
    (OUTPUT_DIR / "manifest_comparacion_temporal.json").write_text(
        json.dumps(manifest_comparacion_temporal, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
display(pd.DataFrame([{
    "exportacion": "preparada" if EXPORTAR_RESULTADOS else "desactivada",
    "ruta": str(OUTPUT_DIR),
    "tablas": len(tablas_salida), "figuras": len(figuras_temporales),
    "estado_metodologico": manifest_comparacion_temporal["estado"],
}]))